In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import datetime as dt
import pyarrow as pa
import pyarrow.parquet as pq

from pygei.seges import alunos

In [2]:
saveFolder = Path (r'N:\05 - Arquivos Pessoais\Vinicius\painel_alertas_otimizacoes\dados')

NUM_ANO_LETIVO = '2026'

available_dates = [date for date in alunos.available_dates() if date.year == int(NUM_ANO_LETIVO)]

In [3]:
dfEmpilhado = pd.read_parquet(saveFolder / 'EMPILHADO_MATRICULAS.parquet')

DatasMatricula = dfEmpilhado.data_referencia.unique()

In [4]:
DatasFaltantes =  set(available_dates) - set(DatasMatricula)

DatasFaltantes

set()

In [ ]:
colunas_alvo = [
    'id_aluno', 'nm_aluno', 'data_nascimento', 'inep_escola', 
    'nm_escola', 'nm_regional', 'id_ano_letivo', 'num_ano_letivo', 'dt_matricula', 
    'dt_enturmacao', 'situacao_enturmacao', 'situacao_matricula', 'tipo_atendimento',
    'fl_deficiencia', 'dc_deficiencia', 'cpf', 'dc_cor_raca', "nome_ano_escolaridade",
    "ano_escolaridade", "nome_turma", "dc_turno",
]

acumulado_base = []

acumulado_base.append(dfEmpilhado)

for date in DatasFaltantes:

    alteracao = 

    df = alunos.load(date.year, date.month, date.day)

    df = df.seges.ativas()

    df = df[colunas_alvo]

    df.num_ano_letivo = df.num_ano_letivo.astype(str)

    df = df.seges.por_ano(NUM_ANO_LETIVO)

    numero_ano_letivo = [ano_letivo for ano_letivo in df.num_ano_letivo.unique() if 'MAPES' not in ano_letivo or 'CEET' not in ano_letivo]

    df = df[df.num_ano_letivo.isin(numero_ano_letivo)]

    df['data_referencia'] = date

    acumulado_base.append(df)

In [ ]:
if DatasFaltantes:
    dfmatricula = pd.concat(acumulado_base)

    table = pa.Table.from_pandas(dfmatricula)

    saveFile = os.path.join(saveFolder, f'EMPILHADO_MATRICULAS.parquet')

    pq.write_table(table, saveFile)

ArrowMemoryError: realloc of size 536870912 failed